# Codification 1 : optimisation des modèles de septembre

Troisième notebook de machine learning de l'essai (journal : [`tableau_resultats.md`](tableau_resultats.md)). Il reprend [`01b_ml_septembre_log.ipynb`](01b_ml_septembre_log.ipynb) à l'identique (données, découpage, montants dus, payés et plafond au logarithme signé, puis standardisés pour la régression logistique et le MLP) et **pousse les cinq modèles au bout de leurs réglages**.

**Hypothèse testée, une seule** : la codification 1 est un **code d'attente**, posé par la banque à la place d'une vraie codification qu'elle n'a pas encore tranchée. `dpnm` n'est utilisé à aucun moment. Le test reste de côté.

**Méthode** :
- **validation croisée** à 5 plis sur le train (21 049 clients) : chaque réglage apprend sur 4 parts et il est noté sur la cinquième, cinq fois ;
- **boucle d'optimisation** : pour chaque paramètre ordonné (profondeur, régularisation, taille des feuilles…), une échelle de valeurs ; la grille essaie la valeur courante et ses deux voisines. Si le meilleur réglage tombe au bord de la grille, la boucle se recentre sur lui et recommence ; elle s'arrête quand le meilleur réglage est à l'intérieur de la grille (stable) ou au bout de l'échelle, ou après `MAX_TOURS` tours (10) ;
- **surapprentissage limité à 7 % relatif** : un réglage n'est retenu que si (score en train - score en validation) / score en train <= 7 % ;
- **alerte de variance** : ⚠️ si l'écart-type des scores des 5 plis dépasse 2 points.

## 1. Données

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display

# Racine du projet : premier dossier parent qui contient 'data'
current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())

df = pd.read_csv(root_dir / "data" / "creditcard_pret_ingestion.csv")
df = df.drop(columns="dpnm")   # jamais utilisé dans ce notebook
PAY = [f"PAY_{n}" for n in range(1, 7)]
print(f"Clients du niveau 0 : {len(df)} ; colonnes : {df.shape[1]} (dpnm retirée) ; PAY_1 = 1 : {int((df['PAY_1'] == 1).sum())}")

# Codifications 1 d'avril à août : leurs clients ont tous PAY_1 = 1, ils restent donc hors du train et du test
anciens_1 = (df[PAY[1:]] == 1).any(axis=1)
assert (df.loc[anciens_1, "PAY_1"] == 1).all()
print(f"Clients avec une codification 1 d'avril à août : {int(anciens_1.sum())}, tous codifiés 1 aussi en septembre")

Clients du niveau 0 : 30000 ; colonnes : 24 (dpnm retirée) ; PAY_1 = 1 : 3688
Clients avec une codification 1 d'avril à août : 28, tous codifiés 1 aussi en septembre


## 2. Septembre : ce que la banque voyait, et ce qu'on cherche à retrouver

**Cible** : la codification de `PAY_1`, **telle quelle**, de -2 à 8 (une classe par codification, sans regrouper les retards) : le niveau du retard fait partie de ce que le modèle doit comprendre. Un essai avec les retards regroupés viendra après, seulement si le résultat est peu probant.

**Variables** : les colonnes du dataset d'origine (niveau 0), sauf `ID` (un identifiant), `PAY_1` (la cible), `dpnm` et les données personnelles ; **aucune colonne ajoutée** :
- les codifications d'avril à août (`PAY_2` à `PAY_6`), telles quelles : les 28 clients qui y ont une codification 1 l'ont tous aussi en septembre, ils ne sont donc ni dans le train ni dans le test ;
- les soldes de fin de mois (`BILL_AMT1` à `BILL_AMT6`), les paiements (`PAY_AMT1` à `PAY_AMT6`) et le plafond (`LIMIT_BAL`).

Les données personnelles (`SEX`, `EDUCATION`, `MARRIAGE`, `AGE`) sont retirées : elles ne sont normalement pas pertinentes pour une codification, qui est un statut de paiement.

Ce ML est un **essai indépendant** : il ne reprend aucun choix des autres branches de ML du projet.

**Exemples** : les clients dont `PAY_1` n'est pas un 1, coupés en 80/20 (train / test, `stratify` sur la classe, `random_state=42`). Les `PAY_1` = 1 sont mis de côté : ce sont eux qu'on cherche à trancher.

In [2]:
from sklearn.model_selection import train_test_split

# Variables : colonnes du dataset d'origine (niveau 0), sauf ID (identifiant), PAY_1 (la cible), dpnm et les données personnelles ; aucune colonne ajoutée
PERSONNELLES = ["SEX", "EDUCATION", "MARRIAGE", "AGE"]   # retirées : non pertinentes pour une codification de paiement
NUMERIQUES = ["LIMIT_BAL"] + [f"BILL_AMT{n}" for n in range(1, 7)] + [f"PAY_AMT{n}" for n in range(1, 7)]
CATEGORIELLES = PAY[1:]
VARIABLES = CATEGORIELLES + NUMERIQUES
assert sorted(VARIABLES + ["ID", "PAY_1"] + PERSONNELLES) == sorted(df.columns)
mask_1 = df["PAY_1"] == 1
connus = df[~mask_1]
a_trancher = df[mask_1]

X_train, X_test, y_train, y_test = train_test_split(
    connus[VARIABLES], connus["PAY_1"], test_size=0.2, stratify=connus["PAY_1"], random_state=42)
CLASSES = sorted(connus["PAY_1"].unique())   # toutes les codifications de PAY_1, sauf 1

repartition = pd.DataFrame({
    "Train": y_train.value_counts(),
    "Test": y_test.value_counts(),
}).reindex(CLASSES).fillna(0).astype(int)
repartition.loc["Total"] = repartition.sum()
print(f"PAY_1 connus : {len(connus)} (train {len(X_train)}, test {len(X_test)}) ; PAY_1 = 1 à trancher : {len(a_trancher)}")
print(f"Variables ({len(VARIABLES)}) : {VARIABLES}")
display(repartition)

PAY_1 connus : 26312 (train 21049, test 5263) ; PAY_1 = 1 à trancher : 3688
Variables (18) : ['PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'LIMIT_BAL', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']


,Train,Test
PAY_1,,
-2,2207,552
-1,4549,1137
0,11789,2948
2,2133,534
3,258,64
4,61,15
5,21,5
6,9,2
7,7,2


**Comment lire** : nombre de clients par codification de `PAY_1`, dans le train et dans le test. Les proportions sont les mêmes dans les deux (`stratify`). Le test n'est plus touché jusqu'au critère A.

## 3. Les niveaux de retard d'août à septembre (train)

`PAY_2` compte des retards importants (3 à 8) qui repassent à 2 en septembre, ce qui peut perturber les modèles. Le tableau croise, sur les clients du train, la codification d'août (en ligne) et celle de septembre (en colonne) : les codifications saines (-2, -1, 0) sont réunies, les niveaux de retard sont détaillés.

In [3]:
def libelle_retard(codifications):
    # Niveaux de retard détaillés, codifications saines réunies
    return codifications.astype(int).astype(str).where(codifications >= 2, "sain (-2, -1, 0)")

croisement = pd.crosstab(libelle_retard(X_train["PAY_2"]).rename("Août (PAY_2)"),
                         libelle_retard(y_train).rename("Septembre (PAY_1)"), margins=True, margins_name="Total")
display(croisement)

Septembre (PAY_1),2,3,4,5,6,7,8,"sain (-2, -1, 0)",Total
Août (PAY_2),,,,,,,,,
2,1266,216,0,0,0,0,0,326,1808
3,57,35,46,0,0,0,0,35,173
4,11,6,12,20,0,0,0,4,53
5,0,1,3,0,9,0,0,1,14
6,0,0,0,1,0,7,0,0,8
7,0,0,0,0,0,0,15,0,15
"sain (-2, -1, 0)",799,0,0,0,0,0,0,18179,18978
Total,2133,258,61,21,9,7,15,18545,21049


**Comment lire** : chaque case donne le nombre de clients du train passés de la codification d'août (ligne) à celle de septembre (colonne). La diagonale correspond aux retards gardés au même niveau ; sous la diagonale, à gauche, les retards qui baissent (par exemple de 3 en août à 2 en septembre) ; au-dessus, ceux qui montent. La ligne et la colonne « Total » donnent le nombre de codifications de chaque niveau, en août et en septembre.

## 4. Le score optimisé : la moyenne des quatre familles (choix validé)

- **`"familles équilibrées"`** (retenu) : la moyenne des taux de bonne famille de -2, -1, 0 et 2 et plus. Chaque famille pèse autant : la boucle est poussée à mieux reconnaître les retards, là où tous les modèles butent ;
- **`"bonnes prédictions"`** : la part des clients rangés dans la bonne famille (le score global des notebooks 01 et 01b). Le 0, 56 % des clients, y pèse le plus. Non retenu : les codifications 1 à trancher ne ressemblent pas au train (environ la moitié suivent un retard en août, contre environ 10 % des `PAY_1` connus) ; un modèle réglé sur ce score pencherait vers les codifications saines, majoritaires dans le train. Il reste affiché dans les tableaux de la section 8.

In [4]:
# ==============================================================================
# Score optimisé (choix validé) : "familles équilibrées" ; autre possibilité : "bonnes prédictions"
# ==============================================================================
SCORE = "familles équilibrées"
SURAPPRENTISSAGE_MAX = 7.0   # écart relatif train - validation, en %
VARIANCE_MAX = 2.0           # écart-type des 5 plis, en points
MAX_TOURS = 10               # tours de la boucle, par modèle

import time
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, make_scorer
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

FAMILLES = [-2, -1, 0, 2]
NOMS_FAMILLES = {-2: "-2", -1: "-1", 0: "0", 2: "2 et plus"}

def famille_pay(codifications):
    # Codification -> famille : -2, -1, 0 gardés, 2 et plus réunis en 2
    v = np.asarray(codifications).ravel().astype(int)
    return np.where(v >= 2, 2, v)

def score_familles(vrai, predit):
    v, p = famille_pay(vrai), famille_pay(predit)
    return balanced_accuracy_score(v, p) if SCORE == "familles équilibrées" else (v == p).mean()

assert SCORE in ("familles équilibrées", "bonnes prédictions")
scoreur = make_scorer(score_familles)

def bonnes_predictions(vrai, predit):
    # Pourcentage de bonnes prédictions (bonne famille), en tout et dans chaque famille
    v, p = famille_pay(vrai), famille_pay(predit)
    resultat = {"global": (v == p).mean() * 100}
    for f in FAMILLES:
        resultat[f] = (p[v == f] == f).mean() * 100
    return resultat

# Logarithme signé des montants dus, payés et du plafond, comme dans 01b
MONTANTS_DUS_PAYES = [f"BILL_AMT{n}" for n in range(1, 7)] + [f"PAY_AMT{n}" for n in range(1, 7)] + ["LIMIT_BAL"]

def log_montants(X):
    X = X.copy()
    X[MONTANTS_DUS_PAYES] = np.sign(X[MONTANTS_DUS_PAYES]) * np.log1p(np.abs(X[MONTANTS_DUS_PAYES]))
    return X

def etape_log():
    return FunctionTransformer(log_montants)

def preprocesseur_lineaire():
    # Codifications : une colonne par valeur ; nombres (déjà au logarithme) : mise à l'échelle standard
    return ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORIELLES),
        ("nombres", StandardScaler(), NUMERIQUES),
    ])

def preprocesseur_arbres():
    # Catégories : une colonne par valeur ; nombres tels quels (déjà au logarithme)
    return ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORIELLES),
    ], remainder="passthrough")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f"Score optimisé : {SCORE} ; surapprentissage max {SURAPPRENTISSAGE_MAX} % relatif ; alerte si écart-type des plis > {VARIANCE_MAX} points")

Score optimisé : familles équilibrées ; surapprentissage max 7.0 % relatif ; alerte si écart-type des plis > 2.0 points


## 5. Les réglages explorés

Pour chaque modèle : les **échelles** (paramètres ordonnés, explorés par la boucle autour d'une valeur de départ, la valeur par défaut de la bibliothèque quand elle est dans l'échelle) et les **choix** (paramètres sans ordre, tous essayés à chaque tour). Les autres paramètres gardent leur valeur par défaut, sauf ceux indiqués en commentaire.

- **Régression logistique** : force de la régularisation `C` (petit = forte régularisation) ; poids des classes (`class_weight` : aucun, ou équilibré pour compenser les codifications rares).
- **Arbre de décision** : profondeur maximale, taille minimale des feuilles (grande = règles plus générales) ; poids des classes ; critère de coupe (`gini` ou `entropy`).
- **RandomForest** : nombre d'arbres (départ à 100, la valeur par défaut), profondeur maximale, taille minimale des feuilles ; part des variables essayées à chaque coupe (`max_features`) ; poids des classes.
- **MLP** : taille des couches cachées, régularisation `alpha` ; arrêt anticipé (`early_stopping`) pour éviter d'apprendre par cœur.
- **CatBoost** : réglé comme dans le machine learning de la deuxième itération (même pipeline que les arbres, codifications en une colonne par valeur ; tous les cœurs ; 1 000 itérations ; départ de la grille à la profondeur 4, `l2_leaf_reg` 10, `learning_rate` 0,05). La boucle explore la profondeur, le `learning_rate` et `l2_leaf_reg` ; poids des classes : `Balanced` ou `SqrtBalanced` (plus doux), toujours actifs, CatBoost refusant la valeur « aucun » passée explicitement.

In [5]:
ESPACES = {
    "Régression logistique": {
        "modele": make_pipeline(etape_log(), preprocesseur_lineaire(), LogisticRegression(max_iter=5000)),
        "n_jobs": -1,
        "echelles": {"logisticregression__C": [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100, 300]},
        "depart": {"logisticregression__C": 1},
        "choix": {"logisticregression__class_weight": [None, "balanced"]},
    },
    "Arbre de décision": {
        "modele": make_pipeline(etape_log(), preprocesseur_arbres(), DecisionTreeClassifier(random_state=42)),
        "n_jobs": -1,
        "echelles": {"decisiontreeclassifier__max_depth": [3, 4, 5, 6, 8, 10, 12, 15, 20, None],
                     "decisiontreeclassifier__min_samples_leaf": [1, 2, 5, 10, 20, 50, 100, 200]},
        "depart": {"decisiontreeclassifier__max_depth": 5, "decisiontreeclassifier__min_samples_leaf": 1},
        "choix": {"decisiontreeclassifier__class_weight": [None, "balanced"],
                  "decisiontreeclassifier__criterion": ["gini", "entropy"]},
    },
    "RandomForest": {
        "modele": make_pipeline(etape_log(), preprocesseur_arbres(), RandomForestClassifier(random_state=42, n_jobs=-1)),
        "n_jobs": -1,
        "echelles": {"randomforestclassifier__n_estimators": [100, 200, 300, 500, 1000],
                     "randomforestclassifier__max_depth": [6, 8, 10, 12, 15, 20, 25, None],
                     "randomforestclassifier__min_samples_leaf": [1, 2, 3, 5, 10, 20, 50]},
        "depart": {"randomforestclassifier__n_estimators": 100, "randomforestclassifier__max_depth": None, "randomforestclassifier__min_samples_leaf": 1},
        "choix": {"randomforestclassifier__max_features": ["sqrt", 0.5],
                  "randomforestclassifier__class_weight": [None, "balanced_subsample"]},
    },
    "MLP": {
        "modele": make_pipeline(etape_log(), preprocesseur_lineaire(), MLPClassifier(max_iter=500, early_stopping=True, random_state=42)),
        "n_jobs": -1,
        "echelles": {"mlpclassifier__hidden_layer_sizes": [(32,), (64,), (100,), (128, 64), (256, 128), (256, 128, 64)],
                     "mlpclassifier__alpha": [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1]},
        "depart": {"mlpclassifier__hidden_layer_sizes": (100,), "mlpclassifier__alpha": 1e-4},
        "choix": {},
    },
    "CatBoost": {
        # Réglé comme dans le machine learning de la deuxième itération : même pipeline que les arbres (codifications en une colonne par valeur),
        # tous les cœurs, pas de fichiers écrits ; poids des classes toujours actifs (CatBoost refuse auto_class_weights=None passé explicitement)
        "modele": make_pipeline(etape_log(), preprocesseur_arbres(),
                                CatBoostClassifier(loss_function="MultiClass", iterations=1000, random_seed=42, verbose=0,
                                                   thread_count=-1, allow_writing_files=False)),
        "n_jobs": -1,
        "echelles": {"catboostclassifier__depth": [2, 3, 4, 5, 6, 7, 8, 10],
                     "catboostclassifier__learning_rate": [0.01, 0.02, 0.03, 0.05, 0.1, 0.2],
                     "catboostclassifier__l2_leaf_reg": [1, 3, 5, 10, 15, 30, 60]},
        "depart": {"catboostclassifier__depth": 4, "catboostclassifier__learning_rate": 0.05, "catboostclassifier__l2_leaf_reg": 10},
        "choix": {"catboostclassifier__auto_class_weights": ["Balanced", "SqrtBalanced"]},
    },
}
for nom, e in ESPACES.items():
    for p, v in e["depart"].items():
        assert v in e["echelles"][p], (nom, p)
print("Modèles :", list(ESPACES))

Modèles : ['Régression logistique', 'Arbre de décision', 'RandomForest', 'MLP', 'CatBoost']


## 6. La boucle d'optimisation

Pour chaque modèle, tour après tour : GridSearch en validation croisée (5 plis) sur la grille courante, puis choix du meilleur réglage **parmi ceux qui surapprennent de 7 % relatif au plus**. Pour chaque modèle, une ligne : son meilleur réglage admis, son score en validation croisée et la raison de l'arrêt de la boucle.

In [6]:
def fenetre(echelle, position):
    # La valeur courante et ses deux voisines dans l'échelle
    return echelle[max(0, position - 1): position + 2]

def libelle(params):
    return ", ".join(f"{k.split('__')[1]}={v}" for k, v in params.items())

def optimiser(nom, espace):
    positions = {p: espace["echelles"][p].index(v) for p, v in espace["depart"].items()}
    lignes, raison, debut = [], f"{MAX_TOURS} tours atteints", time.perf_counter()
    for tour in range(1, MAX_TOURS + 1):
        grille = {p: fenetre(espace["echelles"][p], i) for p, i in positions.items()} | espace["choix"]
        recherche = GridSearchCV(espace["modele"], grille, scoring=scoreur, cv=cv, n_jobs=espace["n_jobs"], return_train_score=True)
        recherche.fit(X_train, y_train, **espace.get("fit_params", {}))
        r = recherche.cv_results_
        res = pd.DataFrame({
            "tour": tour,
            "params": r["params"],
            "validation": r["mean_test_score"] * 100,
            "écart-type des plis": r["std_test_score"] * 100,
            "train": r["mean_train_score"] * 100,
        })
        res["surapprentissage (% relatif)"] = (res["train"] - res["validation"]) / res["train"] * 100
        res["admis"] = res["surapprentissage (% relatif)"] <= SURAPPRENTISSAGE_MAX
        lignes.append(res)
        admis = res[res["admis"]]
        if admis.empty:
            raison = f"tour {tour} : aucun réglage sous {SURAPPRENTISSAGE_MAX} % de surapprentissage"
            break
        meilleur = admis.loc[admis["validation"].idxmax(), "params"]
        nouvelles = {p: espace["echelles"][p].index(meilleur[p]) for p in positions}
        if nouvelles == positions:
            raison = f"tour {tour} : meilleur réglage stable (à l'intérieur de la grille ou au bout de l'échelle)"
            break
        positions = nouvelles
    historique = pd.concat(lignes, ignore_index=True)
    return historique, raison, round(time.perf_counter() - debut)

resultats = {}
for nom, espace in ESPACES.items():
    historique, raison, duree = optimiser(nom, espace)
    resultats[nom] = {"historique": historique, "raison": raison, "duree": duree}
    admis = historique[historique["admis"]]
    if admis.empty:
        print(f"{nom} : aucun réglage admis ; arrêt : {raison} ({duree} s)")
    else:
        m = admis.loc[admis["validation"].idxmax()]
        print(f"{nom} : {m['validation']:.2f} % ± {m['écart-type des plis']:.2f} ({libelle(m['params'])}) ; arrêt : {raison} ({duree} s)")

Régression logistique : 86.05 % ± 0.65 (C=10, class_weight=balanced) ; arrêt : tour 3 : meilleur réglage stable (à l'intérieur de la grille ou au bout de l'échelle) (21 s)
Arbre de décision : 85.88 % ± 0.62 (class_weight=None, criterion=entropy, max_depth=5, min_samples_leaf=1) ; arrêt : tour 1 : meilleur réglage stable (à l'intérieur de la grille ou au bout de l'échelle) (2 s)
RandomForest : 86.84 % ± 0.58 (class_weight=balanced_subsample, max_depth=20, max_features=0.5, min_samples_leaf=2, n_estimators=100) ; arrêt : tour 3 : meilleur réglage stable (à l'intérieur de la grille ou au bout de l'échelle) (327 s)
MLP : 86.51 % ± 0.49 (alpha=1e-06, hidden_layer_sizes=(256, 128)) ; arrêt : tour 3 : meilleur réglage stable (à l'intérieur de la grille ou au bout de l'échelle) (36 s)
CatBoost : 87.01 % ± 0.63 (auto_class_weights=Balanced, depth=5, l2_leaf_reg=5, learning_rate=0.1) ; arrêt : tour 2 : meilleur réglage stable (à l'intérieur de la grille ou au bout de l'échelle) (1164 s)


## 7. Synthèse : le meilleur réglage de chaque modèle

Pour chaque modèle, le meilleur réglage admis (surapprentissage de 7 % relatif au plus) sur l'ensemble des tours, avec son score en validation croisée, l'écart-type de ses 5 plis, son score en train et son surapprentissage. Alertes : ⚠️ variance si l'écart-type des plis dépasse 2 points ; ⚠️ aucun réglage admis.

In [7]:
lignes, meilleurs = [], {}
for nom, r in resultats.items():
    h = r["historique"]
    admis = h[h["admis"]]
    if admis.empty:
        ligne = h.loc[h["surapprentissage (% relatif)"].idxmin()]
        alerte = "⚠️ aucun réglage admis (le moins surappris est affiché)"
    else:
        ligne = admis.loc[admis["validation"].idxmax()]
        alerte = f"⚠️ variance ({ligne['écart-type des plis']:.1f} points)" if ligne["écart-type des plis"] > VARIANCE_MAX else "—"
    meilleurs[nom] = ligne["params"]
    lignes.append({"Modèle": nom, "Réglage": libelle(ligne["params"]),
                   f"Validation ({SCORE}, %)": f"{ligne['validation']:.2f} ± {ligne['écart-type des plis']:.2f}",
                   "Train (%)": round(ligne["train"], 2),
                   "Surapprentissage (% relatif)": round(ligne["surapprentissage (% relatif)"], 2),
                   "Alerte": alerte, "Tours": int(h["tour"].max()), "Arrêt": r["raison"], "Durée (s)": r["duree"]})
display(pd.DataFrame(lignes).set_index("Modèle"))

,Réglage,"Validation (familles équilibrées, %)",Train (%),Surapprentissage (% relatif),Alerte,Tours,Arrêt,Durée (s)
Modèle,,,,,,,,
Régression logistique,"C=10, class_weight=balanced",86.05 ± 0.65,86.23,0.22,—,3,tour 3 : meilleur réglage stable (à l'intérieu...,21
Arbre de décision,"class_weight=None, criterion=entropy, max_dept...",85.88 ± 0.62,86.08,0.23,—,1,tour 1 : meilleur réglage stable (à l'intérieu...,2
RandomForest,"class_weight=balanced_subsample, max_depth=20,...",86.84 ± 0.58,92.69,6.31,—,3,tour 3 : meilleur réglage stable (à l'intérieu...,327
MLP,"alpha=1e-06, hidden_layer_sizes=(256, 128)",86.51 ± 0.49,87.31,0.92,—,3,tour 3 : meilleur réglage stable (à l'intérieu...,36
CatBoost,"auto_class_weights=Balanced, depth=5, l2_leaf_...",87.01 ± 0.63,89.77,3.08,—,2,tour 2 : meilleur réglage stable (à l'intérieu...,1164


## 8. Les modèles réglés, sur tout le train (prédictions hors pli)

Chaque modèle, avec son meilleur réglage, prédit chaque client du train une fois, par le pli qui ne l'a pas appris (`cross_val_predict`, mêmes 5 plis). Les tableaux reprennent ceux des notebooks 01 et 01b, mais sur les 21 049 clients du train au lieu des 4 210 de la validation : les petits groupes (sorties du retard, niveaux de retard élevés) y sont cinq fois plus nombreux.

In [8]:
predictions_hors_pli = {}
for nom, params in meilleurs.items():
    modele = ESPACES[nom]["modele"].set_params(**params)
    predictions_hors_pli[nom] = np.asarray(cross_val_predict(modele, X_train, y_train, cv=cv, n_jobs=ESPACES[nom]["n_jobs"], params=ESPACES[nom].get("fit_params", {}))).ravel().astype(int)
    print(f"{nom} : fait")
predictions_hors_pli["Règle « même codification qu'en août »"] = X_train["PAY_2"].to_numpy()
predictions_hors_pli["Toujours 0"] = np.zeros(len(X_train), dtype=int)

lignes = []
for nom, p in predictions_hors_pli.items():
    b = bonnes_predictions(y_train, p)
    lignes.append({"Modèle": nom, "Bonnes prédictions (%)": round(b["global"], 1),
                   **{f"Famille {NOMS_FAMILLES[f]} (%)": round(b[f], 1) for f in FAMILLES}})
display(pd.DataFrame(lignes).set_index("Modèle"))

Régression logistique : fait
Arbre de décision : fait
RandomForest : fait
MLP : fait
CatBoost : fait


,Bonnes prédictions (%),Famille -2 (%),Famille -1 (%),Famille 0 (%),Famille 2 et plus (%)
Modèle,,,,,
Régression logistique,89.0,94.9,84.7,93.3,71.2
Arbre de décision,90.5,95.0,86.9,96.4,65.3
RandomForest,91.1,95.1,88.4,96.4,67.4
MLP,91.1,95.1,87.6,96.9,66.4
CatBoost,89.6,94.9,89.2,92.7,71.3
Règle « même codification qu'en août »,89.7,92.9,81.6,96.9,68.1
Toujours 0,56.0,0.0,0.0,100.0,0.0


**Comment lire** : une ligne par modèle réglé, sur les prédictions hors pli du train ; les deux dernières lignes sont les points de comparaison. Mêmes mesures que dans les notebooks 01 et 01b.

In [9]:
famille_aout, famille_sept = famille_pay(X_train["PAY_2"]), famille_pay(y_train)
change = famille_aout != famille_sept
TYPES_CHANGEMENT = {
    "Sortie du retard (2 et plus en août, sain en septembre)": (famille_aout == 2) & (famille_sept != 2),
    "Retard maintenu (2 et plus en août et en septembre ; pas un changement)": (famille_aout == 2) & (famille_sept == 2),
    "Nouveau retard (sain en août, 2 et plus en septembre)": (famille_aout != 2) & (famille_sept == 2),
    "Changement entre codifications saines (par exemple 0 → -1)": change & (famille_aout != 2) & (famille_sept != 2),
    "Tous les clients qui changent": change,
}
lignes = []
for type_changement, masque in TYPES_CHANGEMENT.items():
    resultat = {"Type de changement": type_changement, "Clients": int(masque.sum())}
    for nom, p in predictions_hors_pli.items():
        if nom == "Toujours 0":
            continue
        resultat[f"{nom} : retrouvés (%)"] = round((famille_pay(p)[masque] == famille_sept[masque]).mean() * 100, 1)
    lignes.append(resultat)
display(pd.DataFrame(lignes).set_index("Type de changement"))

,Clients,Régression logistique : retrouvés (%),Arbre de décision : retrouvés (%),RandomForest : retrouvés (%),MLP : retrouvés (%),CatBoost : retrouvés (%),Règle « même codification qu'en août » : retrouvés (%)
Type de changement,,,,,,,
"Sortie du retard (2 et plus en août, sain en septembre)",366,38.3,52.2,74.6,77.3,68.9,0.0
Retard maintenu (2 et plus en août et en septembre ; pas un changement),1705,98.7,95.8,98.1,97.3,98.2,100.0
"Nouveau retard (sain en août, 2 et plus en septembre)",799,12.6,0.0,2.0,0.5,13.8,0.0
Changement entre codifications saines (par exemple 0 → -1),998,9.4,9.9,12.2,20.1,17.7,0.0
Tous les clients qui changent,2163,15.5,13.4,19.0,22.6,24.9,0.0


**Comment lire** : comme dans les notebooks 01 et 01b, une ligne par type de changement entre août et septembre, avec la part des clients rangés dans leur vraie famille de septembre ; la règle naïve est à 0 % sur les changements et à 100 % sur les retards maintenus.

In [10]:
NIVEAUX = [c for c in CLASSES if c >= 2]
vrai = y_train.to_numpy()
lignes = []
for niveau in NIVEAUX:
    resultat = {"Codification": niveau, "Vraies (train)": int((vrai == niveau).sum())}
    for nom, p in predictions_hors_pli.items():
        if nom == "Toujours 0":
            continue
        resultat[f"{nom} : attribuées"] = int((p == niveau).sum())
        resultat[f"{nom} : dont justes"] = int(((p == niveau) & (vrai == niveau)).sum())
    lignes.append(resultat)
display(pd.DataFrame(lignes).set_index("Codification"))

,Vraies (train),Régression logistique : attribuées,Régression logistique : dont justes,Arbre de décision : attribuées,Arbre de décision : dont justes,RandomForest : attribuées,RandomForest : dont justes,MLP : attribuées,MLP : dont justes,CatBoost : attribuées,CatBoost : dont justes,Règle « même codification qu'en août » : attribuées,Règle « même codification qu'en août » : dont justes
Codification,,,,,,,,,,,,,
2,2133,1401,886,1810,1281,1466,1228,1502,1248,1766,1216,1808,1266
3,258,810,165,0,0,200,90,126,62,380,120,173,35
4,61,144,45,0,0,85,29,61,26,97,35,53,12
5,21,31,13,0,0,32,12,34,16,28,9,14,0
6,9,15,6,0,0,13,8,14,9,12,8,8,0
7,7,12,7,0,0,8,7,7,7,8,7,15,0
8,15,15,15,0,0,16,15,15,15,15,15,0,0


**Comment lire** : une ligne par niveau de retard ; « vraies » : clients du train qui ont ce niveau en septembre ; « attribuées » : clients à qui le modèle réglé le donne (hors pli) ; « dont justes » : ceux qui l'ont vraiment.